In [ ]:
# Pothole detection training with YOLOv11

!pip install ultralytics kagglehub coremltools -q

import os, random, shutil
from sklearn.model_selection import train_test_split
import kagglehub

# Download dataset
path = kagglehub.dataset_download("atulyakumar98/pothole-detection-dataset")
print("Dataset path:", path)

# Create dataset folders
base_dir = "/content/yolo_dataset"
images_train = os.path.join(base_dir, "images/train")
images_val = os.path.join(base_dir, "images/val")
labels_train = os.path.join(base_dir, "labels/train")
labels_val = os.path.join(base_dir, "labels/val")

for d in [images_train, images_val, labels_train, labels_val]:
    os.makedirs(d, exist_ok=True)

# Get image paths
pothole_dir = os.path.join(path, "potholes")
normal_dir = os.path.join(path, "normal")

pothole_images = [
    os.path.join(pothole_dir, f)
    for f in os.listdir(pothole_dir)
    if f.lower().endswith((".jpg", ".png"))
]

normal_images = [
    os.path.join(normal_dir, f)
    for f in os.listdir(normal_dir)
    if f.lower().endswith((".jpg", ".png"))
]

print(f"Found {len(pothole_images)} pothole images and {len(normal_images)} normal images")

all_images = pothole_images + normal_images
labels = [0] * len(pothole_images) + [1] * len(normal_images)

# Train/validation split
train_imgs, val_imgs, train_labels, val_labels = train_test_split(
    all_images,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print(f"Train images: {len(train_imgs)}")
print(f"Validation images: {len(val_imgs)}")

# Create YOLO labels
def prepare_data(images, labels, img_dest, label_dest):
    for img_path, lbl in zip(images, labels):

        fname = os.path.basename(img_path)
        shutil.copy(img_path, os.path.join(img_dest, fname))

        label_path = os.path.join(
            label_dest,
            fname.rsplit(".", 1)[0] + ".txt"
        )

        with open(label_path, "w") as f:
            f.write(f"{lbl} 0.5 0.5 1 1\n")

prepare_data(train_imgs, train_labels, images_train, labels_train)
prepare_data(val_imgs, val_labels, images_val, labels_val)

print("Dataset prepared")

# Create data.yaml
data_yaml = f"""
train: {images_train}
val: {images_val}

nc: 2
names: ['pothole', 'normal']
"""

with open("/content/data.yaml", "w") as f:
    f.write(data_yaml)

print("data.yaml created")

from ultralytics import YOLO

# Load base YOLOv11 model
model = YOLO("yolo11n.pt")

# Train model
results = model.train(
    data="/content/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    save=True,
    device=0,

    # augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,

    # optimizer
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,

    # loss weights
    box=7.5,
    cls=0.5,
    dfl=1.5,

    name="pothole_yolo11_improved",
    exist_ok=True,
    pretrained=True,
    verbose=True,
)

# Validation
print("\nRunning validation...")
metrics = model.val()

print(f"mAP50: {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")

# Export CoreML model
print("\nExporting CoreML model...")
coreml_model_path = model.export(format="coreml", nms=True)

print("CoreML model exported:", coreml_model_path)

# Save to Drive
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/runs/detect/pothole_yolo11_improved/weights/best.mlpackage /content/drive/MyDrive/best_improved.mlpackage

print("Model copied to Google Drive")

# Show training graph
from IPython.display import Image, display

print("\nTraining results:")
display(Image('/content/runs/detect/pothole_yolo11_improved/results.png'))

print("\nTraining complete")
print("Download best_improved.mlpackage and use it in Xcode")

In [ ]:
from ultralytics import YOLO

# Load trained model
model = YOLO('/content/runs/detect/pothole_yolo11_improved/weights/best.pt')

# Run validation again to regenerate confusion matrix
metrics = model.val()

# Download confusion matrix
from google.colab import files
files.download('/content/runs/detect/pothole_yolo11_improved2/confusion_matrix.png')

# Larger confusion matrix display
import matplotlib.pyplot as plt
from PIL import Image as PILImage

cm_path = '/content/runs/detect/pothole_yolo11_improved2/confusion_matrix.png'
cm_image = PILImage.open(cm_path)

plt.figure(figsize=(7, 7))
plt.imshow(cm_image)
plt.axis('off')

plt.title("Confusion Matrix", fontsize=22, fontweight='bold', pad=20)
plt.xlabel("Predicted Labels", fontsize=18, labelpad=10)
plt.ylabel("True Labels", fontsize=18, labelpad=10)

plt.tight_layout()
plt.show()